<a href="https://colab.research.google.com/github/ys5591/Yunsu-Shin/blob/main/Final_Practice_tmaze_glmhmm_working_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Step 0. Installation / environment setup
# ============================================================

# In Colab, run this cell first.
# The author's GLM-HMM code uses a custom ssm fork with HMM_TO and input-driven transitions.
# This cell avoids uninstalling core packages unless the import fails.

import sys, subprocess, importlib.util, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def pip_install(cmd):
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Core analysis packages
try:
    import numpy, pandas, scipy, matplotlib
except Exception:
    pip_install([sys.executable, "-m", "pip", "install", "numpy", "pandas", "scipy", "matplotlib", "openpyxl"])

# Author's custom ssm fork
try:
    import ssm
    has_hmm_to = hasattr(ssm, "HMM_TO")
    print("ssm is installed. Has HMM_TO:", has_hmm_to)
    if not has_hmm_to:
        raise ImportError("Installed ssm does not have HMM_TO.")
except Exception as e:
    print("Installing author's custom ssm fork because:", repr(e))
    pip_install([sys.executable, "-m", "pip", "install", "autograd"])
    pip_install([sys.executable, "-m", "pip", "install", "git+https://github.com/Zeinab-Mohammadi/ssm.git"])
    import ssm
    print("Installed custom ssm. Has HMM_TO:", hasattr(ssm, "HMM_TO"))

# Clone the paper/GitHub framework for reference.
REPO_DIR = Path("/content/glm-hmm_final") if IN_COLAB else Path("./glm-hmm_final")
if not REPO_DIR.exists():
    try:
        subprocess.run(["git", "clone", "https://github.com/Zeinab-Mohammadi/glm-hmm_final.git", str(REPO_DIR)], check=True)
    except Exception as e:
        print("Could not clone repo automatically. This is okay if you only use the self-contained cells below.")
        print("Reason:", repr(e))

GLMHMM_CODE_DIR = REPO_DIR / "GLMHMM_and_GLM_frameworks"
if GLMHMM_CODE_DIR.exists():
    sys.path.append(str(GLMHMM_CODE_DIR))
print("Repo directory:", REPO_DIR)
print("Framework directory exists:", GLMHMM_CODE_DIR.exists())

Installing author's custom ssm fork because: ModuleNotFoundError("No module named 'ssm'")
Running: /usr/bin/python3 -m pip install autograd
Running: /usr/bin/python3 -m pip install git+https://github.com/Zeinab-Mohammadi/ssm.git
Installed custom ssm. Has HMM_TO: True
Repo directory: /content/glm-hmm_final
Framework directory exists: True


In [2]:
# ============================================================
# Step 1. Imports, paths, and settings
# ============================================================

import os, re, json, math, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize

# ---------- File paths ----------
# Your local folder in Finder / File Explorer is:
#   Yunsu-Personal > Desktop > Columbia Research
# In Python, this is represented as Path.home() / "Desktop" / "Columbia Research".
# Example:
#   Windows: C:/Users/Yunsu-Personal/Desktop/Columbia Research
#   Mac:     /Users/Yunsu-Personal/Desktop/Columbia Research

LOCAL_PROJECT_DIR = Path.home() / "Desktop" / "Columbia Research"
COLAB_PROJECT_DIR = Path("/content/Columbia Research")
SANDBOX_PROJECT_DIR = Path("/mnt/data")  # only for this ChatGPT-generated copy

if IN_COLAB:
    PROJECT_DIR = COLAB_PROJECT_DIR
elif SANDBOX_PROJECT_DIR.exists():
    # This lets the notebook run inside this ChatGPT sandbox for checking.
    # When you run it on your computer, change this line to: PROJECT_DIR = LOCAL_PROJECT_DIR
    PROJECT_DIR = LOCAL_PROJECT_DIR if LOCAL_PROJECT_DIR.exists() else SANDBOX_PROJECT_DIR
else:
    PROJECT_DIR = LOCAL_PROJECT_DIR

DATA_DIR = PROJECT_DIR
RESULTS_DIR = PROJECT_DIR / "glm_hmm_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Primary file you uploaded. Keep this exact filename in Columbia Research.
CSV_FILENAME = "working_memory_long_format_clean(2).csv"
CLEAN_CSV_PATH = DATA_DIR / CSV_FILENAME

# Fallback filenames/locations, useful if you rename the file or run in Colab.
# The notebook will try the new file first, then older copies.
CSV_CANDIDATES = [
    CLEAN_CSV_PATH,
    DATA_DIR / "working_memory_long_format_clean(2).csv",
    DATA_DIR / "working_memory_long_format_clean(1).csv",
    DATA_DIR / "working_memory_long_format_clean.csv",
    Path("working_memory_long_format_clean(2).csv"),
    Path("working_memory_long_format_clean(1).csv"),
    Path("working_memory_long_format_clean.csv"),
    Path("/mnt/data/working_memory_long_format_clean(2).csv"),
    Path("/mnt/data/working_memory_long_format_clean(1).csv"),
    Path("/mnt/data/working_memory_long_format_clean.csv"),
    Path("/content/working_memory_long_format_clean(2).csv"),
    Path("/content/working_memory_long_format_clean(1).csv"),
    Path("/content/working_memory_long_format_clean.csv"),
]

if IN_COLAB and not any(p.exists() for p in CSV_CANDIDATES):
    from google.colab import files
    print("Please upload working_memory_long_format_clean(2).csv")
    uploaded = files.upload()

for candidate in CSV_CANDIDATES:
    if candidate.exists():
        CLEAN_CSV_PATH = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the clean CSV. Put working_memory_long_format_clean(2).csv in: "
        f"{LOCAL_PROJECT_DIR}\n"
        "Expected breadcrumb: Yunsu-Personal > Desktop > Columbia Research"
    )

print("Project folder:", PROJECT_DIR)
print("Clean CSV path:", CLEAN_CSV_PATH)
print("Results folder:", RESULTS_DIR)

# ---------- Modeling settings ----------
RANDOM_SEED = 0
N_FOLDS = 5

# Paper used tau=4 for exponential filtering of transition covariates.
# We keep tau=4 as the default but you can test tau=2,4,8 later.
TAU = 4.0

# For first pass, compare K=1 GLM, K=2 GLM-HMM, and K=3 GLM-HMM.
# K=4 is possible but may overfit because each mouse has only ~270 trials.
K_LIST = [2, 3]

N_INITS = 3          # increase to 5-10 for final analysis
N_EM_ITERS = 150     # increase to 300 for final analysis, matching the paper
PRIOR_SIGMA = 100.0
TRANSITION_ALPHA = 1.0

Project folder: /content/Columbia Research
Clean CSV path: working_memory_long_format_clean.csv
Results folder: /content/Columbia Research/glm_hmm_results


In [3]:
# ============================================================
# Step 2. Load the clean long-format T-maze CSV
# ============================================================

# This notebook now starts from the clean CSV file rather than re-parsing the Excel file.
# Required file location on your computer:
#   Yunsu-Personal > Desktop > Columbia Research > working_memory_long_format_clean(2).csv

print("Loading:", CLEAN_CSV_PATH)
df_long = pd.read_csv(CLEAN_CSV_PATH)

# Standardize date column if present.
if "session_date" in df_long.columns:
    df_long["session_date"] = pd.to_datetime(df_long["session_date"], errors="coerce")
elif "session_date_raw" in df_long.columns:
    df_long["session_date"] = pd.to_datetime(
        df_long["session_date_raw"],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )
else:
    df_long["session_date"] = pd.NaT

# Make sure numeric columns are numeric.
for col in ["testing_num", "delay_from_label", "session_index", "trial_in_session", "delay_s",
            "correct_cumulative_original", "choice_right", "sample_right", "correct_side_right", "trial_correct"]:
    if col in df_long.columns:
        df_long[col] = pd.to_numeric(df_long[col], errors="coerce")

# Clean side labels.
for col in ["sample_side", "choice"]:
    if col in df_long.columns:
        df_long[col] = df_long[col].astype(str).str.lower().str.strip()

required_cols = [
    "mouse_id", "session_id", "trial_in_session", "delay_s",
    "sample_side", "choice", "trial_correct", "correct_side_right"
]
missing = [c for c in required_cols if c not in df_long.columns]
if missing:
    raise ValueError(f"Missing required columns in clean CSV: {missing}")

print("Long-format shape:", df_long.shape)
print("Mice:", df_long["mouse_id"].nunique())
print("Sessions:", df_long["session_id"].nunique())
print("Delay counts:")
display(df_long["delay_s"].value_counts().sort_index())

print("Trials per session:")
display(df_long.groupby("session_id").size().value_counts().sort_index())

display(df_long.head(20))

# Save a standardized copy without changing your original file.
standardized_csv_path = RESULTS_DIR / "working_memory_long_format_clean_standardized.csv"
df_long.to_csv(standardized_csv_path, index=False)
print("Saved standardized copy:", standardized_csv_path)


Loading: working_memory_long_format_clean.csv
Long-format shape: (12150, 17)
Mice: 45
Sessions: 405
Delay counts:


,count
delay_s,
10,4050
30,4079
60,4021


Trials per session:


,count
29,1
30,403
31,1


,mouse_id,testing_label,testing_num,delay_from_label,session_date_raw,session_index,session_id,trial_in_session,delay_s,sample_side,choice,correct_cumulative_original,choice_right,sample_right,correct_side_right,trial_correct,session_date
0,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,0,10,left,right,1,1,0,1,1,2025-08-18
1,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,1,10,right,left,2,0,1,0,1,2025-08-18
2,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,2,10,left,left,2,0,0,1,0,2025-08-18
3,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,3,10,left,right,3,1,0,1,1,2025-08-18
4,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,4,10,right,left,4,0,1,0,1,2025-08-18
5,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,5,10,left,right,5,1,0,1,1,2025-08-18
6,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,6,10,right,right,5,1,1,0,0,2025-08-18
7,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,7,10,right,left,6,0,1,0,1,2025-08-18
8,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,8,10,right,left,7,0,1,0,1,2025-08-18
9,Sd43,Testing1-10,1,10,45887,0,Sd43_Testing1-10_000,9,10,left,right,8,1,0,1,1,2025-08-18


Saved standardized copy: /content/Columbia Research/glm_hmm_results/working_memory_long_format_clean_standardized.csv


In [4]:
# ============================================================
# Step 3. Validate and derive DNMS trial variables
# ============================================================

df_model = df_long.copy()

# Clean side labels
for col in ["sample_side", "choice"]:
    df_model[col] = df_model[col].astype(str).str.lower().str.strip()

valid_sides = {"left", "right"}
bad_sample = df_model.loc[~df_model["sample_side"].isin(valid_sides)]
bad_choice = df_model.loc[~df_model["choice"].isin(valid_sides)]

if len(bad_sample) > 0:
    display(bad_sample.head())
    raise ValueError("Some sample/order values are not left/right.")

if len(bad_choice) > 0:
    display(bad_choice.head())
    raise ValueError("Some choice values are not left/right.")

# Sort by real experimental order.
df_model = df_model.sort_values(
    ["mouse_id", "session_date", "delay_s", "testing_num", "session_index", "trial_in_session"]
).reset_index(drop=True)

# Binary coding
# left = 0, right = 1 for output y
df_model["choice_right"] = (df_model["choice"] == "right").astype(int)
df_model["sample_right"] = (df_model["sample_side"] == "right").astype(int)

# Signed coding
# left = -1, right = +1
df_model["choice_signed"] = df_model["choice_right"] * 2 - 1
df_model["sample_signed"] = df_model["sample_right"] * 2 - 1

# DNMS rule: correct choice is opposite of sample/order side.
df_model["correct_side_signed"] = -df_model["sample_signed"]
df_model["correct_side_right"] = (df_model["correct_side_signed"] == 1).astype(int)

# Use trial-level correctness derived from rule, not the original cumulative Correct column.
df_model["trial_correct"] = (df_model["choice_signed"] == df_model["correct_side_signed"]).astype(int)
df_model["reward_signed"] = df_model["trial_correct"] * 2 - 1

# Optional sanity: compare derived cumulative correctness to original cumulative column.
df_model["derived_correct_cumulative"] = (
    df_model.groupby("session_id")["trial_correct"].cumsum()
)

cumulative_mismatch = (
    pd.to_numeric(df_model["correct_cumulative_original"], errors="coerce")
    != df_model["derived_correct_cumulative"]
)
print("Rows where original cumulative Correct differs from derived cumulative correctness:", int(cumulative_mismatch.sum()))
print("This can happen if the original cumulative column was typed/reset manually. We will use derived trial_correct.")

summary_by_delay = (
    df_model.groupby("delay_s")
    .agg(
        n_trials=("trial_correct", "size"),
        accuracy=("trial_correct", "mean"),
        p_right_choice=("choice_right", "mean"),
        p_right_correct_side=("correct_side_right", "mean")
    )
    .reset_index()
)
display(summary_by_delay)

display(df_model.head())

Rows where original cumulative Correct differs from derived cumulative correctness: 9
This can happen if the original cumulative column was typed/reset manually. We will use derived trial_correct.


,delay_s,n_trials,accuracy,p_right_choice,p_right_correct_side
0,10,4050,0.756049,0.489877,0.499753
1,30,4079,0.689630,0.467517,0.499387
2,60,4021,0.608058,0.490177,0.499130


,mouse_id,testing_label,testing_num,delay_from_label,session_date_raw,session_index,session_id,trial_in_session,delay_s,sample_side,...,choice_right,sample_right,correct_side_right,trial_correct,session_date,choice_signed,sample_signed,correct_side_signed,reward_signed,derived_correct_cumulative
0,Sd101,Testing1-10,1,10,45941,360,Sd101_Testing1-10_360,0,10,right,...,1,1,0,0,2025-10-11,1,1,-1,-1,0
1,Sd101,Testing1-10,1,10,45941,360,Sd101_Testing1-10_360,1,10,right,...,0,1,0,1,2025-10-11,-1,1,-1,1,1
2,Sd101,Testing1-10,1,10,45941,360,Sd101_Testing1-10_360,2,10,right,...,0,1,0,1,2025-10-11,-1,1,-1,1,2
3,Sd101,Testing1-10,1,10,45941,360,Sd101_Testing1-10_360,3,10,left,...,1,0,1,1,2025-10-11,1,-1,1,1,3
4,Sd101,Testing1-10,1,10,45941,360,Sd101_Testing1-10_360,4,10,left,...,0,0,1,0,2025-10-11,-1,-1,1,-1,3


In [5]:
# ============================================================
# Step 4. Build observation and transition regressors
# ============================================================

def zscore(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)
    if sd == 0 or np.isnan(sd):
        return np.zeros_like(x, dtype=float)
    return (x - np.nanmean(x)) / sd

def add_prev_within_session(df, source_col, new_col, fill_value=0.0):
    df[new_col] = (
        df.groupby("session_id", sort=False)[source_col]
        .shift(1)
        .fillna(fill_value)
        .astype(float)
    )
    return df

def exponential_filter_by_session(df, source_col, new_col, tau=4.0):
    '''
    Exponential filter within each session.
    alpha = exp(-1/tau). Larger tau means longer memory.

    We apply this to previous-trial variables so the transition input at trial t
    reflects history available before or at that transition.
    '''
    alpha = np.exp(-1.0 / tau)
    out = np.zeros(len(df), dtype=float)

    for _, idx in df.groupby("session_id", sort=False).groups.items():
        idx = np.asarray(idx)
        x = df.loc[idx, source_col].to_numpy(dtype=float)
        filt = np.zeros_like(x, dtype=float)
        running = 0.0
        for t in range(len(x)):
            running = alpha * running + (1 - alpha) * x[t]
            filt[t] = running
        out[idx] = filt

    df[new_col] = out
    return df

def add_temporal_bases(df):
    '''
    Paper used 3 temporal bases for the first 100 trials to model warm-up.
    Your sessions are ~30 trials, so these bases span each full session.
    '''
    progress = (
        df.groupby("session_id", sort=False)["trial_in_session"]
        .transform(lambda s: s / max(s.max(), 1))
        .astype(float)
    )

    # Three simple linearly independent temporal bases.
    df["basis_early"] = zscore(1.0 - progress)
    df["basis_middle"] = zscore(4.0 * progress * (1.0 - progress))
    df["basis_late"] = zscore(progress)
    df["trial_progress"] = progress
    return df

df_model = add_prev_within_session(df_model, "choice_signed", "prev_choice_signed")
df_model = add_prev_within_session(df_model, "correct_side_signed", "prev_correct_side_signed")
df_model = add_prev_within_session(df_model, "reward_signed", "prev_reward_signed")

df_model = exponential_filter_by_session(df_model, "prev_choice_signed", "filt_prev_choice", tau=TAU)
df_model = exponential_filter_by_session(df_model, "prev_correct_side_signed", "filt_prev_correct_side", tau=TAU)
df_model = exponential_filter_by_session(df_model, "prev_reward_signed", "filt_prev_reward", tau=TAU)

df_model = add_temporal_bases(df_model)

df_model["delay_z"] = zscore(df_model["delay_s"])
df_model["correct_side_x_delay_z"] = df_model["correct_side_signed"] * df_model["delay_z"]

# Observation GLM:
# Predicts current left/right choice. This is where "strategy" weights are defined.
# The last column is an explicit intercept/bias, matching the author's HMM_TO observation format.
obs_regressor_names = [
    "correct_side_signed",       # DNMS rule / working-memory evidence
    "prev_choice_signed",        # perseveration / repeat tendency
    "prev_reward_signed",        # win-stay/lose-shift tendency
    "delay_z",                   # global delay effect
    "correct_side_x_delay_z",    # whether rule-following changes with delay
    "obs_bias"                   # right/left side bias
]
df_model["obs_bias"] = 1.0

# Transition GLM:
# Predicts transitions between hidden states.
# These are intentionally different from observation regressors, following the paper.
trans_regressor_names = [
    "filt_prev_choice",
    "filt_prev_correct_side",
    "filt_prev_reward",
    "delay_z",
    "basis_early",
    "basis_middle",
    "basis_late",
    "trans_bias"
]
df_model["trans_bias"] = 1.0

obs_mat = df_model[obs_regressor_names].to_numpy(dtype=float)
trans_mat = df_model[trans_regressor_names].to_numpy(dtype=float)
output = df_model["choice_right"].to_numpy(dtype=int).reshape(-1, 1)
session = df_model["session_id"].astype(str).to_numpy()

assert obs_mat.shape[0] == trans_mat.shape[0] == output.shape[0] == session.shape[0]
assert set(np.unique(output[:, 0])).issubset({0, 1})
assert np.isfinite(obs_mat).all()
assert np.isfinite(trans_mat).all()

print("obs_mat:", obs_mat.shape)
print("trans_mat:", trans_mat.shape)
print("output:", output.shape)
print("session:", session.shape)

print("\nObservation regressors:")
display(pd.DataFrame(obs_mat[:10], columns=obs_regressor_names))

print("\nTransition regressors:")
display(pd.DataFrame(trans_mat[:10], columns=trans_regressor_names))


obs_mat: (12150, 6)
trans_mat: (12150, 8)
output: (12150, 1)
session: (12150,)

Observation regressors:


,correct_side_signed,prev_choice_signed,prev_reward_signed,delay_z,correct_side_x_delay_z,obs_bias
0,-1.0,0.0,0.0,-1.134319,1.134319,1.0
1,-1.0,1.0,-1.0,-1.134319,1.134319,1.0
2,-1.0,-1.0,1.0,-1.134319,1.134319,1.0
3,1.0,-1.0,1.0,-1.134319,-1.134319,1.0
4,1.0,1.0,1.0,-1.134319,-1.134319,1.0
5,-1.0,-1.0,-1.0,-1.134319,1.134319,1.0
6,-1.0,1.0,-1.0,-1.134319,1.134319,1.0
7,1.0,-1.0,1.0,-1.134319,-1.134319,1.0
8,-1.0,-1.0,-1.0,-1.134319,1.134319,1.0
9,1.0,-1.0,1.0,-1.134319,-1.134319,1.0



Transition regressors:


,filt_prev_choice,filt_prev_correct_side,filt_prev_reward,delay_z,basis_early,basis_middle,basis_late,trans_bias
0,0.000000,0.000000,0.000000,-1.134319,1.675185,-2.023512,-1.675185,1.0
1,0.221199,-0.221199,-0.221199,-1.134319,1.559638,-1.604906,-1.559638,1.0
2,-0.048929,-0.393469,0.048929,-1.134319,1.444091,-1.216201,-1.444091,1.0
3,-0.259305,-0.527633,0.259305,-1.134319,1.328544,-0.857396,-1.328544,1.0
4,0.019252,-0.189722,0.423146,-1.134319,1.212996,-0.528491,-1.212996,1.0
5,-0.206206,0.073443,0.108347,-1.134319,1.097449,-0.229487,-1.097449,1.0
6,0.060606,-0.164001,-0.136818,-1.134319,0.981902,0.039617,-0.981902,1.0
7,-0.173999,-0.348924,0.114645,-1.134319,0.866355,0.278820,-0.866355,1.0
8,-0.356710,-0.050543,-0.131913,-1.134319,0.750808,0.488123,-0.750808,1.0
9,-0.499005,-0.260562,0.118465,-1.134319,0.635260,0.667525,-0.635260,1.0


In [6]:
# ============================================================
# Step 5. Make 5-fold session-level cross-validation map
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

session_info = (
    df_model[["mouse_id", "session_id", "delay_s", "testing_num"]]
    .drop_duplicates()
    .sort_values(["mouse_id", "delay_s", "testing_num", "session_id"])
    .reset_index(drop=True)
)

fold_records = []

# Assign folds within each mouse so the train/test split is not dominated by a few mice.
for mouse_id, sub in session_info.groupby("mouse_id", sort=False):
    sess = sub["session_id"].to_numpy().copy()
    rng.shuffle(sess)
    for i, sid in enumerate(sess):
        fold_records.append([sid, i % N_FOLDS])

fold_mapping_session = np.array(fold_records, dtype=object)

display(pd.DataFrame(fold_mapping_session, columns=["session_id", "fold"]).head(15))
print("Fold counts:")
display(pd.DataFrame(fold_mapping_session, columns=["session_id", "fold"])["fold"].value_counts().sort_index())

# Save model-ready files in the same general structure as the GitHub code.
np.savez(DATA_DIR / "combined_all_mice.npz", obs_mat, trans_mat, output, session)
np.savez(DATA_DIR / "fold_mapping_session_all_mice.npz", fold_mapping_session)
np.savez(
    DATA_DIR / "regressor_names.npz",
    obs_regressor_names=np.array(obs_regressor_names),
    trans_regressor_names=np.array(trans_regressor_names)
)
df_model.to_csv(DATA_DIR / "working_memory_model_ready_metadata.csv", index=False)

print("Saved model-ready arrays to:", DATA_DIR)

,session_id,fold
0,Sd101_Testing2-30_364,0
1,Sd101_Testing3-30_365,1
2,Sd101_Testing3-10_362,2
3,Sd101_Testing1-60_366,3
4,Sd101_Testing1-30_363,4
5,Sd101_Testing3-60_368,0
6,Sd101_Testing2-60_367,1
7,Sd101_Testing1-10_360,2
8,Sd101_Testing2-10_361,3
9,Sd104_Testing3-10_371,0


Fold counts:


,count
fold,
0,90
1,90
2,90
3,90
4,45


Saved model-ready arrays to: /content/Columbia Research


In [7]:
# ============================================================
# Step 6. Fit 1-state Bernoulli GLM baseline with 5-fold CV
# ============================================================

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logistic_nll_and_grad(w, X, y, prior_sigma=100.0):
    logits = X @ w
    nll = np.sum(np.logaddexp(0, logits) - y * logits)
    nll += 0.5 * np.sum((w / prior_sigma) ** 2)

    p = sigmoid(logits)
    grad = X.T @ (p - y)
    grad += w / (prior_sigma ** 2)
    return nll, grad

def fit_logistic_glm(X_train, y_train, prior_sigma=100.0):
    w0 = np.zeros(X_train.shape[1])
    result = minimize(
        fun=lambda w: logistic_nll_and_grad(w, X_train, y_train, prior_sigma)[0],
        x0=w0,
        jac=lambda w: logistic_nll_and_grad(w, X_train, y_train, prior_sigma)[1],
        method="L-BFGS-B",
        options={"maxiter": 2000, "ftol": 1e-10, "gtol": 1e-6}
    )
    w = result.x
    train_ll = -logistic_nll_and_grad(w, X_train, y_train, prior_sigma)[0]
    return w, train_ll, result

def eval_logistic_glm(w, X, y):
    logits = X @ w
    p = np.clip(sigmoid(logits), 1e-9, 1 - 1e-9)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    acc = np.mean((p >= 0.5).astype(int) == y)
    return ll, acc

X_obs = obs_mat.copy()
y = output[:, 0].astype(int)

glm_summary_rows = []

for fold in range(N_FOLDS):
    print("\n" + "=" * 70)
    print("1-state GLM fold", fold)
    print("=" * 70)

    train_sessions = fold_mapping_session[fold_mapping_session[:, 1].astype(int) != fold, 0].astype(str)
    test_sessions = fold_mapping_session[fold_mapping_session[:, 1].astype(int) == fold, 0].astype(str)

    train_idx = np.isin(session, train_sessions)
    test_idx = np.isin(session, test_sessions)

    X_train, y_train = X_obs[train_idx], y[train_idx]
    X_test, y_test = X_obs[test_idx], y[test_idx]

    w_hat, train_ll, opt = fit_logistic_glm(X_train, y_train, prior_sigma=PRIOR_SIGMA)
    test_ll, test_acc = eval_logistic_glm(w_hat, X_test, y_test)

    p_right_train = np.clip(np.mean(y_train), 1e-9, 1 - 1e-9)
    baseline_test_ll = np.sum(y_test * np.log(p_right_train) + (1 - y_test) * np.log(1 - p_right_train))
    bits_over_baseline = (test_ll - baseline_test_ll) / (len(y_test) * np.log(2))

    print("Converged:", opt.success, "| Test LL/trial:", test_ll / len(y_test), "| Test acc:", test_acc)
    print(pd.Series(w_hat, index=obs_regressor_names))

    # Save GLM initialization in a shape convenient for HMM_TO: 1 x 1 x M_obs
    fold_dir = RESULTS_DIR / "Model" / "glm_#state=1" / f"fld_num={fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    glm_weights_for_hmm = w_hat.reshape(1, 1, -1)
    np.savez(fold_dir / "important_params_iter_0.npz", train_ll, glm_weights_for_hmm)

    glm_summary_rows.append({
        "K": 1,
        "fold": fold,
        "n_train_trials": int(train_idx.sum()),
        "n_test_trials": int(test_idx.sum()),
        "train_ll": float(train_ll),
        "test_ll": float(test_ll),
        "test_ll_per_trial": float(test_ll / len(y_test)),
        "test_accuracy": float(test_acc),
        "baseline_test_ll": float(baseline_test_ll),
        "bits_per_trial_over_baseline": float(bits_over_baseline),
        **{f"w_{name}": float(val) for name, val in zip(obs_regressor_names, w_hat)}
    })

glm_summary = pd.DataFrame(glm_summary_rows)
glm_summary_path = RESULTS_DIR / "glm_1state_cv_summary.csv"
glm_summary.to_csv(glm_summary_path, index=False)

print("\nSaved:", glm_summary_path)
display(glm_summary)
display(glm_summary[["test_ll_per_trial", "test_accuracy", "bits_per_trial_over_baseline"]].mean())


1-state GLM fold 0
Converged: True | Test LL/trial: -0.6086117688716078 | Test acc: 0.6878933728248797
correct_side_signed       0.795021
prev_choice_signed        0.048347
prev_reward_signed       -0.030620
delay_z                   0.038923
correct_side_x_delay_z   -0.258970
obs_bias                 -0.076864
dtype: float64

1-state GLM fold 1
Converged: True | Test LL/trial: -0.6195417566550079 | Test acc: 0.6802519451648759
correct_side_signed       0.790814
prev_choice_signed        0.067405
prev_reward_signed       -0.025687
delay_z                   0.003133
correct_side_x_delay_z   -0.290356
obs_bias                 -0.071830
dtype: float64

1-state GLM fold 2
Converged: True | Test LL/trial: -0.6164293219756598 | Test acc: 0.6855555555555556
correct_side_signed       0.809434
prev_choice_signed        0.021067
prev_reward_signed       -0.029636
delay_z                  -0.013103
correct_side_x_delay_z   -0.293908
obs_bias                 -0.065147
dtype: float64

1-state GLM 

,K,fold,n_train_trials,n_test_trials,train_ll,test_ll,test_ll_per_trial,test_accuracy,baseline_test_ll,bits_per_trial_over_baseline,w_correct_side_signed,w_prev_choice_signed,w_prev_reward_signed,w_delay_z,w_correct_side_x_delay_z,w_obs_bias
0,1,0,9449,2701,-5815.034548,-1643.860388,-0.608612,0.687893,-1871.110607,0.121382,0.795021,0.048347,-0.030620,0.038923,-0.258970,-0.076864
1,1,1,9451,2699,-5785.489825,-1672.143201,-0.619542,0.680252,-1869.202479,0.105334,0.790814,0.067405,-0.025687,0.003133,-0.290356,-0.071830
2,1,2,9450,2700,-5795.333257,-1664.359169,-0.616429,0.685556,-1869.328092,0.109521,0.809434,0.021067,-0.029636,-0.013103,-0.293908,-0.065147
3,1,3,9450,2700,-5819.933166,-1637.605797,-0.606521,0.691852,-1869.204577,0.123751,0.783401,0.060463,-0.027367,0.007513,-0.271491,-0.065674
4,1,4,10800,1350,-6608.025068,-849.570970,-0.629312,0.671852,-935.556311,0.091889,0.801275,0.063924,-0.012768,0.018297,-0.290702,-0.080698


,0
test_ll_per_trial,-0.616083
test_accuracy,0.683481
bits_per_trial_over_baseline,0.110375


In [8]:
# ============================================================
# Step 7. GLM-HMM helper functions
# ============================================================

import ssm

def segment_by_session(obs_arr, trans_arr, y_arr, session_arr, mask_arr=None):
    '''Split trial-level arrays into a list of session sequences.'''
    inputs_obs, inputs_trans, datas, masks, session_ids = [], [], [], [], []

    unique_sessions = pd.Series(session_arr).drop_duplicates().to_numpy()
    counter = 0

    for sid in unique_sessions:
        idx = np.where(session_arr == sid)[0]
        inputs_obs.append(np.asarray(obs_arr[idx, :], dtype=float))
        inputs_trans.append(np.asarray(trans_arr[idx, :], dtype=float))
        datas.append(np.asarray(y_arr[idx, :], dtype=int).reshape(-1, 1))
        if mask_arr is None:
            masks.append(np.ones((len(idx), 1), dtype=bool))
        else:
            masks.append(np.asarray(mask_arr[idx], dtype=bool).reshape(-1, 1))
        session_ids.append(sid)
        counter += len(idx)

    assert counter == obs_arr.shape[0], "Some trials were not assigned to a session."
    return inputs_obs, inputs_trans, datas, masks, session_ids

def make_hmm(K, M_obs, M_trans, C=2, D=1, prior_sigma=100.0, transition_alpha=1.0):
    return ssm.HMM_TO(
        K,
        D,
        M_trans=M_trans,
        M_obs=M_obs,
        observations="input_driven_obs_diff_inputs",
        observation_kwargs=dict(C=C, prior_sigma=prior_sigma),
        transitions="inputdrivenalt",
        transition_kwargs=dict(prior_sigma=prior_sigma, alpha=transition_alpha, kappa=0)
    )

def initialize_hmm_from_glm(model, glm_weights, K, seed=0, noise_scale=0.05, zero_transition_weights=True):
    '''
    glm_weights should have shape 1 x 1 x M_obs.
    We tile it across K states and add small noise to break symmetry.
    '''
    rng = np.random.default_rng(seed)

    if glm_weights.ndim == 1:
        glm_weights = glm_weights.reshape(1, 1, -1)
    if glm_weights.shape[1] != 1:
        # If saved with class baseline + class 1, keep class 1 only.
        glm_weights = glm_weights[:, 1:2, :]

    M_obs = glm_weights.shape[-1]
    obs_init = np.tile(glm_weights, (K, 1, 1))
    obs_init = obs_init + rng.normal(0, noise_scale, size=obs_init.shape)
    model.observations.params = obs_init

    if zero_transition_weights:
        try:
            # In the author's code, params[1] are input-driven transition weights.
            model.transitions.params[1][...] = np.zeros_like(model.transitions.params[1])
            print("Transition input weights initialized to zero.")
        except Exception as e:
            print("Could not zero transition input weights; continuing with default transition init.")
            print("Reason:", repr(e))

    return model

def hmm_log_probability(model, datas, obs_inputs, trans_inputs, masks=None):
    '''
    Compatibility wrapper because different ssm forks expose slightly different argument names.
    '''
    attempts = [
        lambda: model.log_probability(datas, transition_input=trans_inputs, observation_input=obs_inputs, masks=masks),
        lambda: model.log_probability(datas, transition_input=trans_inputs, observation_input=obs_inputs),
        lambda: model.log_probability(datas, inputs=obs_inputs, masks=masks),
        lambda: model.log_probability(datas, inputs=obs_inputs),
    ]
    last_error = None
    for fn in attempts:
        try:
            return fn()
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not compute HMM log_probability. Last error: {repr(last_error)}")

def fit_one_glmhmm(K, fold, iter_id, num_iters=N_EM_ITERS):
    '''
    Fit one K-state GLM-HMM for one fold and one initialization.
    '''
    train_sessions = fold_mapping_session[fold_mapping_session[:, 1].astype(int) != fold, 0].astype(str)
    test_sessions = fold_mapping_session[fold_mapping_session[:, 1].astype(int) == fold, 0].astype(str)

    train_idx = np.isin(session, train_sessions)
    test_idx = np.isin(session, test_sessions)

    obs_train, trans_train, y_train, session_train = obs_mat[train_idx], trans_mat[train_idx], output[train_idx], session[train_idx]
    obs_test, trans_test, y_test, session_test = obs_mat[test_idx], trans_mat[test_idx], output[test_idx], session[test_idx]

    inputs_train, inputs_trans_train, datas_train, masks_train, session_ids_train = segment_by_session(
        obs_train, trans_train, y_train, session_train
    )
    inputs_test, inputs_trans_test, datas_test, masks_test, session_ids_test = segment_by_session(
        obs_test, trans_test, y_test, session_test
    )

    M_obs = obs_mat.shape[1]
    M_trans = trans_mat.shape[1]

    glm_init_path = RESULTS_DIR / "Model" / "glm_#state=1" / f"fld_num={fold}" / "important_params_iter_0.npz"
    glm_container = np.load(glm_init_path, allow_pickle=True)
    glm_weights = glm_container["arr_1"]

    model = make_hmm(K, M_obs, M_trans, C=2, D=1, prior_sigma=PRIOR_SIGMA, transition_alpha=TRANSITION_ALPHA)
    model = initialize_hmm_from_glm(model, glm_weights, K=K, seed=iter_id, noise_scale=0.05)

    print(f"Fitting K={K}, fold={fold}, init={iter_id}, train sessions={len(session_ids_train)}, test sessions={len(session_ids_test)}")

    lls = model.fit(
        datas_train,
        transition_input=inputs_trans_train,
        observation_input=inputs_train,
        train_masks=masks_train,
        method="em",
        num_iters=num_iters,
        initialize=False,
        tolerance=1e-4
    )

    test_ll = hmm_log_probability(model, datas_test, inputs_test, inputs_trans_test, masks_test)
    test_n = int(y_test.shape[0])
    train_n = int(y_train.shape[0])

    # Baseline: p(right) from training fold.
    p_right_train = np.clip(np.mean(y_train[:, 0]), 1e-9, 1 - 1e-9)
    baseline_test_ll = np.sum(
        y_test[:, 0] * np.log(p_right_train) + (1 - y_test[:, 0]) * np.log(1 - p_right_train)
    )
    bits_over_baseline = (test_ll - baseline_test_ll) / (test_n * np.log(2))

    save_dir = RESULTS_DIR / "Model" / f"glmhmm_#state={K}" / f"fld_num={fold}" / f"iter_{iter_id}"
    save_dir.mkdir(parents=True, exist_ok=True)

    raw_param_path = save_dir / f"glm_hmm_raw_parameters_itr_{iter_id}.npz"
    np.savez(raw_param_path, model.params, lls)

    row = {
        "K": K,
        "fold": fold,
        "iter": iter_id,
        "n_train_trials": train_n,
        "n_test_trials": test_n,
        "n_train_sessions": len(session_ids_train),
        "n_test_sessions": len(session_ids_test),
        "initial_train_ll": float(lls[0]),
        "final_train_ll": float(lls[-1]),
        "train_ll_improvement": float(lls[-1] - lls[0]),
        "test_ll": float(test_ll),
        "test_ll_per_trial": float(test_ll / test_n),
        "baseline_test_ll": float(baseline_test_ll),
        "bits_per_trial_over_baseline": float(bits_over_baseline),
        "raw_param_path": str(raw_param_path)
    }

    return row

In [9]:
# ============================================================
# Step 8. Run GLM-HMM model comparison for K=2 and K=3
# ============================================================

# For a quick smoke test, temporarily use:
# K_LIST = [2]
# N_INITS = 1
# N_EM_ITERS = 25

all_hmm_rows = []

for K in K_LIST:
    for fold in range(N_FOLDS):
        for iter_id in range(N_INITS):
            row = fit_one_glmhmm(K=K, fold=fold, iter_id=iter_id, num_iters=N_EM_ITERS)
            all_hmm_rows.append(row)

            running = pd.DataFrame(all_hmm_rows)
            running_path = RESULTS_DIR / "glmhmm_all_fits_running_summary.csv"
            running.to_csv(running_path, index=False)

hmm_all = pd.DataFrame(all_hmm_rows)
hmm_all_path = RESULTS_DIR / "glmhmm_all_fits_summary.csv"
hmm_all.to_csv(hmm_all_path, index=False)

print("Saved:", hmm_all_path)
display(hmm_all)

Transition input weights initialized to zero.
Fitting K=2, fold=0, init=0, train sessions=315, test sessions=90


  0%|          | 0/150 [00:00<?, ?it/s]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.